

# Lab 3 · Auditing Your ETL Pipeline
### Does your pipeline see what *you* see?

*Solo lab · about 2 hours · Colab · your own Google Cloud bucket (or Dr. Wanik's backup data)*

Your Craigslist pipeline has been running on Google Cloud. Tonight you **audit** it: pull your data into this
notebook, **read** the ads, **measure** what the regex extractor got right and wrong, then **fix** the worst field.

| Part | Time | What you do |
|---|---|---|
| **1. Get it running** | 30 min | Finish the Final GCP Guide F26 if you have not yet |
| **2. Search and see** | 25 min | Your cars land in the **Files panel** on the left. Search them, then see each ad three ways: web page, text, extracted fields |
| **3. Validate the extractor** | 30 min | Ground truth for 5 cars: what the extractor pulled out vs what the ad really says |
| **4. Fix it** | 30 min | Three ways to pull make and model: plain Python (given), **your regex**, and a typo-fixer (stretch) |
| **5. Wrap up** | 5 min | Save your truth table for A07, answer 3 questions, submit |

**First: File → Save a copy in Drive.** This notebook opens from GitHub, so your edits disappear when you close the tab unless you work in your own copy.

**Tonight everyone starts on Dr. Wanik's Hartford backup data** (the box in Part 2a is ticked). When your own pipeline
is running, untick it, type your project ID, and re-run from Part 2a: the whole lab works on your own cars too.

**Stuck?** Ask Claude: paste the cell, the full error, and what you expected.

# Part 1 · Get it running (30 min)

Skip to Part 2 if you already have **5 green checks** and at least one scrape in your bucket.

1. Follow the **[Final GCP Guide F26](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/FINAL_GCP_Guide_F26.ipynb)** (HuskyCT, Week 3.1) through Step 10.
2. In **Cloud Scheduler**, tick the checkbox, then **Force run**, **one at a time**, waiting for **Success** each time:
   1. `craigslist-scraper-hourly`
   2. `extractor-per-listing-hourly`
   3. `materialize-master-hourly`
3. In **Cloud Storage**, open your bucket: you should see `scrapes/` (raw ads) and `structured/` (the extracted fields).

# Part 2 · Search and see (25 min)

## 2a. Pick your data and load it
- **Backup data (start here):** leave `USE_BACKUP_DATA` **ticked**. You get Dr. Wanik's 30 Hartford cars. No sign-in needed.
- **Your own data (later):** **untick** the box, type your project ID, and re-run from this cell to the end. A pop-up asks
  which Google account to use: pick the **same Gmail you used for Google Cloud** (not necessarily your UConn account).
  You get your **newest 30 cars**, newest first.

Switching back and forth is safe: the next cell clears the old copy before it loads the new one.

In [ ]:
USE_BACKUP_DATA = True  #@param {type:"boolean"}
PROJECT_ID = "YOUR-PROJECT-ID"  #@param {type:"string"}
BUCKET_NAME = "YOUR-PROJECT-ID"  #@param {type:"string"}
# Ticked = Dr. Wanik's Hartford backup data. Unticked = your own bucket (BUCKET_NAME is usually the same as PROJECT_ID).

In [ ]:
import io
import json
import os
import re
import shutil
import zipfile
import urllib.request
import pandas as pd

DATA_DIR = "/content/bucket_copy"     # a copy of your bucket, inside this notebook
MAX_CARS = 30                         # your own bucket: copy only your NEWEST 30 cars (fresh ads, fast download)
shutil.rmtree(DATA_DIR, ignore_errors=True)   # start clean, so backup and own data never mix
os.makedirs(DATA_DIR, exist_ok=True)

if USE_BACKUP_DATA:
    SAMPLE_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab3_hartford_sample.zip"
    zip_bytes = urllib.request.urlopen(SAMPLE_URL).read()
    zipfile.ZipFile(io.BytesIO(zip_bytes)).extractall(DATA_DIR)
    print("Loaded Dr. Wanik's Hartford backup data")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    master_key = "structured/datasets/listings_master.csv"
    local_master = os.path.join(DATA_DIR, master_key)
    os.makedirs(os.path.dirname(local_master), exist_ok=True)
    bucket.blob(master_key).download_to_filename(local_master)
    newest = pd.read_csv(local_master, dtype={"post_id": str})
    newest = newest.sort_values("scraped_at", ascending=False).head(MAX_CARS)
    run_folders = []
    for source_txt in newest["source_txt"]:
        local_path = os.path.join(DATA_DIR, source_txt)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        bucket.blob(source_txt).download_to_filename(local_path)
        page_blob = bucket.blob(source_txt.replace(".txt", ".html"))
        if page_blob.exists():
            page_blob.download_to_filename(local_path.replace(".txt", ".html"))
        run_folder = os.path.dirname(source_txt)
        if run_folder not in run_folders:
            run_folders.append(run_folder)
    for run_folder in run_folders:
        index_blob = bucket.blob(run_folder + "/index.csv")
        if index_blob.exists():
            index_blob.download_to_filename(os.path.join(DATA_DIR, run_folder, "index.csv"))
    print("Copied your newest", len(newest), "cars from gs://" + BUCKET_NAME)

cars = pd.read_csv(os.path.join(DATA_DIR, "structured/datasets/listings_master.csv"), dtype={"post_id": str})
if not USE_BACKUP_DATA:
    cars = cars.sort_values("scraped_at", ascending=False).head(MAX_CARS).reset_index(drop=True)

link_tables = []
for folder, subfolders, filenames in os.walk(os.path.join(DATA_DIR, "scrapes")):
    for filename in filenames:
        if filename == "index.csv":
            link_tables.append(pd.read_csv(os.path.join(folder, filename), dtype={"post_id": str}))
links = pd.concat(link_tables, ignore_index=True)
links = links[["post_id", "url"]].drop_duplicates("post_id")
cars = cars.merge(links, on="post_id", how="left")
print(len(cars), "cars loaded")

## 2b. Your cars, one file each (look left!)
The next cell makes a folder called **`my_cars`**. Click the **folder icon** in the left sidebar, open `my_cars`, and
**double-click** any file to read it. These are **your saved copies**: they stay even after a seller deletes the ad.
(With your own bucket, file `01` is your **newest** car.)
- `01_2013_8500_....txt` is the **raw ad** (what the scraper saved)
- `01_2013_8500_....json` is **what the extractor pulled out** of it (plus the link to the live ad)
- `01_2013_8500_....html` is the **web page itself** (the HTML the scraper read), and `....jpg` is a **snapshot** of how
  it looked in a browser (backup data only: Dr. Wanik saved these; one ad was deleted before he could)
- `cars.csv` is the whole table

**Try these two first** (with Dr. Wanik's backup data; with your own cars, hunt for your own examples):
- **File 01** (2013 Sonata): the seller typed `Hyunda Sonata`, and the extractor copied it exactly, typo and all. Right by luck, wrong spelling.
- **File 03** (2013 Elantra): the extractor says the make is `Contact` and the model is `Information`. Open the `.txt` and find where that came from.

In [ ]:
def read_ad_text(source_txt):
    with open(os.path.join(DATA_DIR, source_txt), encoding="utf-8") as f:
        return f.read()


def car_file(i):
    row = cars.iloc[i]
    return str(i + 1).zfill(2) + "_" + str(row["year"]) + "_" + str(row["price"]) + "_" + row["post_id"]


shutil.rmtree("/content/my_cars", ignore_errors=True)
os.makedirs("/content/my_cars", exist_ok=True)
for i in range(len(cars)):
    row = cars.iloc[i]
    stem = "/content/my_cars/" + car_file(i)
    with open(stem + ".txt", "w", encoding="utf-8") as f:
        f.write(read_ad_text(row["source_txt"]))
    extracted = {"post_id": row["post_id"], "url": row["url"], "price": row["price"], "year": row["year"],
                 "make": row["make"], "model": row["model"], "mileage": row["mileage"]}
    with open(stem + ".json", "w", encoding="utf-8") as f:
        json.dump(extracted, f, indent=2, default=str)
    for extension in [".html", ".jpg"]:
        saved_page = os.path.join(DATA_DIR, row["source_txt"].replace(".txt", extension))
        if os.path.exists(saved_page):
            shutil.copy(saved_page, stem + extension)
cars.to_csv("/content/my_cars/cars.csv", index=False)
print("Wrote", len(cars), "cars to /content/my_cars (open the folder icon on the left)")
cars[["post_id", "price", "year", "make", "model", "mileage"]].head(10)

**Look at the `make` and `model` columns.** Some look right. Some do not. Keep that thought: it is the mystery for Part 3.

## 2c. Search your cars
The `make` column is not trustworthy yet, so we search the **raw ad text**. Change `SEARCH_WORD` and re-run.
Try a brand (`toyota`, `honda`, `ford`), a feature (`sunroof`, `4wd`), or a phrase like `clean title`.

Each hit shows the **file number** (the same `01`, `02`, ... as the `my_cars` files on the left) and the **line** where the word was found. Surprised by a hit? Read that line. One Hartford seller pasted a `Keywords:` line full of other brands so the ad shows up in more searches (keyword stuffing). A plain text search can't tell the difference.

In [ ]:
SEARCH_WORD = "toyota"

n_found = 0
for i in range(len(cars)):
    row = cars.iloc[i]
    text = read_ad_text(row["source_txt"])
    spot = text.lower().find(SEARCH_WORD.lower())
    if spot >= 0:
        n_found = n_found + 1
        file_number = str(i + 1).zfill(2)
        line_start = text.rfind("\n", 0, spot) + 1
        line_end = text.find("\n", spot)
        if line_end == -1:
            line_end = len(text)
        found_line = text[line_start:line_end].strip()
        print("file", file_number, "|", row["year"], "$" + str(row["price"]), "| found in:", found_line[:90])
        print("         ", "open on the left: my_cars/" + car_file(i) + ".txt")

print()
print(n_found, "of", len(cars), "ads mention", repr(SEARCH_WORD))

## 2d. One ad, three ways
Every car in your pipeline exists in three forms. The next cell shows them side by side:
1. **The web page**, the way a buyer sees it (Dr. Wanik saved a snapshot of each backup ad; scroll inside it)
2. **The text** your scraper saved (`.txt`): the same page with the formatting stripped out
3. **What the extractor pulled out** (`.json`): five fields, found by regular expressions

**Suggested cars (backup data):**
- **01:** the 2013 Sonata. Find `Hyunda` in all three views
- **03:** the 2013 Elantra. Where did `Contact Information` come from? And look at `title status:` on the web page
- **05:** the 2007 Odyssey. The extractor says the make is `Vernon`. Why?

Then try **11**: that seller deleted the ad the day after it was scraped, so there is no web page anymore. Your `.txt`
copy still has everything, which is why pipelines save their own copy. (With your own data, you see the text and the
extracted fields; open the link in the `.json` if you want the live page.)

In [ ]:
CARS_TO_OPEN = ["01", "03", "05"]     # file numbers from the my_cars folder; change them and re-run

import base64
import html as html_lib
from IPython.display import HTML, display

box = "height:440px;overflow:auto;border:1px solid #ccc;padding:6px;font-size:12px;white-space:pre-wrap;margin:0"
for number in CARS_TO_OPEN:
    i = int(number) - 1
    row = cars.iloc[i]
    text = read_ad_text(row["source_txt"])
    with open("/content/my_cars/" + car_file(i) + ".json", encoding="utf-8") as f:
        extracted_text = f.read()
    shot_path = os.path.join(DATA_DIR, row["source_txt"].replace(".txt", ".jpg"))
    page_path = os.path.join(DATA_DIR, row["source_txt"].replace(".txt", ".html"))
    if os.path.exists(shot_path):
        with open(shot_path, "rb") as f:
            shot = base64.b64encode(f.read()).decode("ascii")
        web_view = '<div style="height:440px;overflow:auto;border:1px solid #ccc"><img src="data:image/jpeg;base64,' + shot + '" style="width:100%"></div>'
    elif os.path.exists(page_path):
        with open(page_path, encoding="utf-8") as f:
            page = f.read()
        page = re.sub(r"(?is)<script.*?</script>", "", page)
        page = re.sub(r"(?i)<head[^>]*>", '<head><base href="https://www.craigslist.org/">', page, count=1)
        web_view = '<iframe srcdoc="' + html_lib.escape(page) + '" style="width:100%;height:440px;border:1px solid #ccc"></iframe>'
    else:
        web_view = '<div style="' + box + '"><b>No saved web page for this car.</b> The ad may be gone. The text copy still has everything.</div>'
    panel = ('<h3>File ' + number + ': ' + str(row["year"]) + ', $' + str(row["price"]) + '</h3>'
             + '<div style="display:flex;gap:8px">'
             + '<div style="flex:2"><b>1. The web page</b>' + web_view + '</div>'
             + '<div style="flex:1"><b>2. The text (.txt)</b><pre style="' + box + '">' + html_lib.escape(text) + '</pre></div>'
             + '<div style="flex:1"><b>3. Extracted fields (.json)</b><pre style="' + box + '">' + html_lib.escape(extracted_text) + '</pre></div>'
             + '</div>')
    display(HTML(panel))

**Write 2 or 3 sentences:** what does the saved text keep from the web page, and what does it lose?

*Your answer:*

# Part 3 · Validate the extractor against ground truth (30 min)

The extractor (Module 3.2) uses regular expressions to turn each raw ad into `price, year, make, model, mileage`.
How accurate is it? You find out the way data teams do: build a small **ground-truth** set by hand, then score the
extractor against it.

## 3a. Record the ground truth for 5 cars
The next cell lists files `01` to `05` and what the extractor pulled out of each. Now check them by hand: in the Files
panel, **double-click each `.txt`** and read the real price, year, make, model and odometer. Near the top of each ad
there is a line that is just the **year**, and the next line is the **make and model** (for example `2011` then
`Honda Accord LX`); further down is `odometer:` then the miles.

**Example (backup data, file 03):** the extractor says `4995, 2013, Contact, Information, 88000`. The ad is a
2013 Hyundai Elantra, so the truth is `"price": 4995, "year": 2013, "make": "hyundai", "model": "elantra", "mileage": 88000`.
Price, year and mileage were right. Make and model were not.

**With the backup data the ground truth is already filled in** (two cells down), so the notebook runs start to
finish. Your job: **check at least two cars** against their `.txt` files. A ground-truth set is only as good as the
person who checked it. With your own data, paste the skeleton and type your own values.

**Rules for the truth table:**
- numbers without `$` or commas (`4450`, not `"$4,450"`)
- make and model in **lowercase**, model = **first word only** (`honda`, `accord`)
- write the **real** make: if the seller typed `Hyunda`, the truth is `hyundai`

In [ ]:
graded = cars.head(5)

for i in range(len(graded)):
    row = graded.iloc[i]
    print("file", str(i + 1).zfill(2), "|", row["post_id"], "| extractor says:", row["price"], row["year"], row["make"], row["model"], row["mileage"])

print()
print("Copy this into the next cell and fill in the TRUE values:")
print("truth = {")
for i in range(len(graded)):
    pid = graded.iloc[i]["post_id"]
    print('    "' + pid + '": {"price": None, "year": None, "make": "", "model": "", "mileage": None},')
print("}")

In [ ]:
# Ground truth for files 01 to 05.
# Backup data: already filled in. CHECK at least two cars against their .txt files on the left.
# Your own data: replace {} below with the skeleton printed above, filled in with YOUR values.
if USE_BACKUP_DATA:
    truth = {
        "24WkQVAv4khn6QWdTBQHf1": {"price": 8500, "year": 2013, "make": "hyundai", "model": "sonata", "mileage": 40000},
        "4ysVPsYd2KTDr6rbicgnoV": {"price": 3800, "year": 2011, "make": "honda", "model": "accord", "mileage": 198000},
        "7dEEfyBPBV7RpUamCJ258v": {"price": 4995, "year": 2013, "make": "hyundai", "model": "elantra", "mileage": 88000},
        "8CsTtVC5zZNU7f9SqFpDDc": {"price": 4900, "year": 2009, "make": "acura", "model": "tl", "mileage": 175000},
        "cHAwqCSEDchnoZSBBn4AvL": {"price": 1800, "year": 2007, "make": "honda", "model": "odyssey", "mileage": 213000},
    }
else:
    truth = {}
print(len(truth), "cars in the ground truth")

## 3b. Measure the extractor's accuracy

In [ ]:
FIELDS = ["price", "year", "make", "model", "mileage"]


def same(extracted_value, true_value, field):
    if field in ["make", "model"]:
        return str(extracted_value).strip().lower() == str(true_value).strip().lower()
    try:
        return float(extracted_value) == float(true_value)
    except (TypeError, ValueError):
        return False


def score(table, truth):
    rows = []
    for field in FIELDS:
        right = 0
        for pid in truth:
            table_row = table[table["post_id"] == pid].iloc[0]
            if same(table_row[field], truth[pid][field], field):
                right = right + 1
        rows.append({"field": field, "correct": right, "out_of": len(truth), "accuracy": right / max(len(truth), 1)})
    return pd.DataFrame(rows)


before = score(cars, truth)
print(before.to_string())

**Which fields does the extractor get right? Which does it miss?**

## 3c. The mystery: what is the make/model regex actually grabbing?
Here is the rule, from **line 41** of <a href="https://github.com/drdave-teaching/myscrapers/blob/main/cloud_function/extractor-per-listing/main.py#L41" target="_blank">cloud_function/extractor-per-listing/main.py</a>
(opens in a new tab; it is used at line 142, inside `parse_listing`):

```python
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
```

In words: *the **first** place on the page where one Capitalized word is followed by another Capitalized word.*
Run the cell to see what it grabs from each of your 5 cars.

In [ ]:
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")

for i in range(len(graded)):
    text = read_ad_text(graded.iloc[i]["source_txt"])
    found = MAKE_MODEL_RE.search(text)
    first_line = text.split("\n")[0]
    print("file", str(i + 1).zfill(2), "| grabs:", found.groups() if found else None, "| page title:", first_line[:70])

Each ad's text starts with the page title the seller typed. If they Capitalized it (`2011 Honda Accord`), the rule
works. If they typed it in lowercase or ALL CAPS, the first two Capitalized words are somewhere else on the page:
a button (`Contact Information`), or even the **town** (`Vernon Rockville`).

**See it in file 03** (open `03_...txt` on the left; Colab shows line numbers):

| Line | Text | What it means |
|---|---|---|
| 30 | `Contact Information:` | the **first** two Capitalized words on the page, so the old regex stops here |
| 32 | `2013 HYUNDAI ELANTRA GLS * ONLY 88K MILES*` | the seller's title, in ALL CAPS, so the old regex skips it |
| 37 | `image 1 of 14` | the photo gallery (close, but ads with one photo have no such line) |
| 39 | `2013` | **the anchor: a line that is only the year** |
| 40 | `hyundai elantra` | **make and model, always on the next line** |
| 41 | `condition:` | the details block (condition, cylinders, drive, fuel, odometer...) starts |

Every ad has that year line right above the make and model, typed by Craigslist's form, not the seller. That is the
reliable spot. The next cell finds it for any file and prints the lines around it:

In [ ]:
CAR_NUMBER = 3     # file 03; try 1, 2, 4 or 5 too

text = read_ad_text(cars.iloc[CAR_NUMBER - 1]["source_txt"])
ad_lines = text.split("\n")
for i in range(len(ad_lines)):
    line = ad_lines[i]
    if len(line) == 4 and line.isdigit() and line[:2] in ["19", "20"]:
        for j in range(max(0, i - 3), i + 5):
            note = ""
            if j == i:
                note = "   <-- the anchor: only the year"
            if j == i + 1:
                note = "   <-- make and model"
            print(str(j + 1).rjust(3), repr(ad_lines[j]) + note)
        break

# Part 4 · Fix it: three ways to get make and model (30 min)

The clue: a line that is **only a 4-digit year**, and the **next line** starts with the **make** then the **model**.
Three ways to use that clue, from easiest to fanciest.

## Way 1 · Plain Python, no regex (given: read it, run it)
Walk the lines. When a line is exactly 4 digits and starts with 19 or 20, split the next line into words.
The test prints all 5 files next to what the extractor said. **Watch files 03 and 04**: the extractor said
`Contact Information` for both. Does Way 1 get them right?

In [ ]:
def way1_make_model(text):
    ad_lines = text.split("\n")
    for i in range(len(ad_lines) - 1):
        line = ad_lines[i].strip()
        if len(line) == 4 and line.isdigit() and (line.startswith("19") or line.startswith("20")):
            words = ad_lines[i + 1].strip().lower().split()
            if len(words) >= 2:
                return words[0], words[1]
    return "", ""


for i in range(len(graded)):
    text = read_ad_text(graded.iloc[i]["source_txt"])
    print("file", str(i + 1).zfill(2), "| extractor said:", graded.iloc[i]["make"], graded.iloc[i]["model"], "| way 1:", way1_make_model(text))

## Way 2 · Your regex: the same idea in ONE line
Write one regular expression that matches the year line, the line break, then captures the **make** (group 1) and the
**model** (group 2). Build it a piece at a time and test at **regex101.com** (pick the Python flavor).

| Piece | Means |
|---|---|
| `^` and `$` with `re.MULTILINE` | start / end of a **line** |
| `(?:19\|20)\d{2}` | a year like 1998 or 2017 (`(?: )` groups without capturing) |
| `\n` | the line break |
| `([A-Za-z-]+)` | capture one word of letters (and hyphens, for `mercedes-benz`) |
| `\s+` | one or more spaces |

<details><summary><b>Hint 1</b> (click)</summary>

Start with just the year line: `^(?:19|20)\d{2}$` matches a line like `2013`. Try it on regex101 with one of your ads.
</details>

<details><summary><b>Hint 2</b></summary>

Replace the `$` with `\n` so the match continues onto the next line: `^(?:19|20)\d{2}\n`
</details>

<details><summary><b>Hint 3</b></summary>

Now capture the first word of the next line: `^(?:19|20)\d{2}\n([A-Za-z-]+)`. You add the space and the second group.
</details>

In [ ]:
BETTER_MAKE_MODEL_RE = re.compile(r"YOUR PATTERN HERE", re.MULTILINE)   # <-- your one line


def way2_make_model(text):
    found = BETTER_MAKE_MODEL_RE.search(text)
    if found is None:
        return "", ""
    return found.group(1).lower(), found.group(2).lower()


for i in range(len(graded)):
    text = read_ad_text(graded.iloc[i]["source_txt"])
    print("file", str(i + 1).zfill(2), "| extractor said:", graded.iloc[i]["make"], graded.iloc[i]["model"], "| way 2:", way2_make_model(text))

## Way 3 · Stretch: fix the typos, flag the rest for review
Neither Way 1 nor Way 2 can know that `hyunda` means `hyundai`. A list of real makes, a few nicknames, and Python's
`difflib` (which finds the closest spelling) can. And when a "make" matches **nothing** on the list (like `contact`),
the honest answer is not a guess: it is **`NEEDS_REVIEW`**, so a person looks at it. That is how real data teams
handle the long tail.

In [ ]:
import difflib
from collections import Counter

KNOWN_MAKES = ["acura", "audi", "bmw", "buick", "cadillac", "chevrolet", "chrysler", "dodge", "fiat", "ford", "gmc",
               "honda", "hyundai", "infiniti", "jaguar", "jeep", "kia", "land", "lexus", "lincoln", "mazda",
               "mercedes-benz", "mercury", "mini", "mitsubishi", "nissan", "pontiac", "porsche", "ram", "saab",
               "saturn", "scion", "subaru", "tesla", "toyota", "volkswagen", "volvo"]
NICKNAMES = {"chevy": "chevrolet", "vw": "volkswagen", "mercedes": "mercedes-benz"}


def clean_make(make):
    make = str(make).strip().lower()
    if make in NICKNAMES:
        make = NICKNAMES[make]
    closest = difflib.get_close_matches(make, KNOWN_MAKES, n=1, cutoff=0.75)
    if len(closest) > 0:
        return closest[0]
    return "NEEDS_REVIEW"


def way3_make_model(text):
    make, model = way1_make_model(text)
    return clean_make(make), model


for i in range(len(graded)):
    text = read_ad_text(graded.iloc[i]["source_txt"])
    print("file", str(i + 1).zfill(2), "| extractor said:", graded.iloc[i]["make"], graded.iloc[i]["model"], "| way 3:", way3_make_model(text))

**Now point the same cleaner at ALL your cars**, twice: once on what the **original extractor** said, once on
Way 3. Look at the most common makes, and at everything flagged `NEEDS_REVIEW`.

In [ ]:
original_makes = Counter()
way3_makes = Counter()
review = []
for i in range(len(cars)):
    original = clean_make(cars.iloc[i]["make"])
    fixed = way3_make_model(read_ad_text(cars.iloc[i]["source_txt"]))[0]
    original_makes[original] = original_makes[original] + 1
    way3_makes[fixed] = way3_makes[fixed] + 1
    if fixed == "NEEDS_REVIEW":
        review.append("file " + str(i + 1).zfill(2) + " | " + str(cars.iloc[i]["make"]) + " " + str(cars.iloc[i]["model"]))

print("Original extractor, cleaned:", original_makes.most_common(8))
print("Way 3:                      ", way3_makes.most_common(8))
print()
print(len(review), "cars still need a human (open their .txt on the left):")
for line in review:
    print("  ", line)

**Make it yours.** Your area sells different cars. Grow `KNOWN_MAKES` and `NICKNAMES`, re-run both cells, and
watch the `NEEDS_REVIEW` list shrink. Paste this prompt into Claude, with your two lists pasted where it says:

> I'm cleaning car makes scraped from Craigslist ads in the Hartford, CT area (change to your area). Here is my Python list
> `KNOWN_MAKES` and my dict `NICKNAMES`: *(paste both)*. Add the car makes commonly sold in the US that are missing, plus the
> nicknames and misspellings sellers often type (like `chevy` → `chevrolet`, `vw` → `volkswagen`, `benz` → `mercedes-benz`).
> Keep everything lowercase. Give me the updated list and dict as Python code I can paste, one item per line, and tell me
> what you added and why.

Then check Claude's work: every `NEEDS_REVIEW` car that is a real make should now be fixed, and `contact` should
**still** be flagged. It is not a car. 🙂

## 4b. Re-score: the original extractor vs your three fixes

In [ ]:
def graded_with(make_model_function):
    fixed = graded.copy()
    for i in range(len(fixed)):
        text = read_ad_text(fixed.iloc[i]["source_txt"])
        make, model = make_model_function(text)
        fixed.iloc[i, fixed.columns.get_loc("make")] = make
        fixed.iloc[i, fixed.columns.get_loc("model")] = model
    return fixed


comparison = before[["field", "accuracy"]].rename(columns={"accuracy": "original"})
comparison["way1_python"] = score(graded_with(way1_make_model), truth)["accuracy"]
comparison["way2_your_regex"] = score(graded_with(way2_make_model), truth)["accuracy"]
comparison["way3_typo_fix"] = score(graded_with(way3_make_model), truth)["accuracy"]
print(comparison.to_string())

**Stretch:** run your regex on **all** your cars. How many ads does it miss completely? Open one: why?

In [ ]:
missed = []
for i in range(len(cars)):
    make, model = way2_make_model(read_ad_text(cars.iloc[i]["source_txt"]))
    if make == "":
        missed.append("my_cars/" + car_file(i) + ".txt")
print(len(missed), "of", len(cars), "ads had no match")
for name in missed:
    print(name)

# Part 5 · Wrap up (5 min)

## Save your truth table (you will use it again in A07)
In A07 you let Gemini read the ads instead of a regex. You will score it on **these same 5 cars**, so save your truth
table now: to your bucket (if you used your own data) and to a file you can download.

In [ ]:
with open("/content/my_cars/truth_table.json", "w", encoding="utf-8") as f:
    json.dump(truth, f, indent=2)
print("Saved /content/my_cars/truth_table.json (right-click it on the left and Download)")

if not USE_BACKUP_DATA:
    client.bucket(BUCKET_NAME).blob("lab3/truth_table.json").upload_from_string(json.dumps(truth, indent=2), content_type="application/json")
    print("Also saved to gs://" + BUCKET_NAME + "/lab3/truth_table.json")

## Three short answers (1 or 2 sentences each)
1. Which field did the extractor get **most** right, and which **least**? Why?
2. Where could **your** regex still break? (Think about how different people post their ads.)
3. In A07, Gemini reads the ad instead of a regex. What will it do **better** than Ways 1 to 3, and what could go **wrong**?

*Your answers:*

---

**Submit:** File → Download → `.ipynb`, then upload it to **Submit Lab 3** on HuskyCT (due Thu Oct 8, 11:59 PM).
Keep your pipeline running: the more hours it scrapes, the better your data for the midterm.